![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Week 2, Day 3 -- Lab 3: Image Search with a Frozen EfficientNet

A pretrained model can turn any image into a short list of numbers -- an
**embedding** -- that captures what's in it. Similar images get similar numbers.

We **freeze** EfficientNet (use it without training -- pure transfer learning),
turn a gallery of photos into embeddings, and find look-alikes with **cosine
similarity**.

## 1) Setup

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, random
from torchvision import datasets
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu" 

## 2) Load a frozen EfficientNet (no training)

We drop its classifier so the model outputs the **embedding** instead of a label.

In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights).to(device)
model.classifier = torch.nn.Identity()   # keep the features, remove the classifier
model.eval();

## 3) Get the images: Caltech-101

101 categories of **animals and everyday objects** -- great for searching look-alikes.

In [ ]:
!curl -L -o caltech101.zip --progress-bar "https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip?download=1"
!unzip -q caltech101.zip
!tar -xzf caltech-101/101_ObjectCategories.tar.gz

## 4) Build a gallery of 800 images

In [ ]:
prep = weights.transforms()
full  = datasets.ImageFolder("101_ObjectCategories", transform=prep)
shown = datasets.ImageFolder("101_ObjectCategories")        # same images, for display
idx = random.sample(range(len(full)), 800)                  # a varied gallery
print(len(full), "images across", len(full.classes), "categories")

## 5) Turn every gallery image into an embedding

In [ ]:
vectors = []
with torch.no_grad():
    for s in range(0, len(idx), 64):
        batch = torch.stack([full[i][0] for i in idx[s:s + 64]]).to(device)
        vectors.append(model(batch).cpu())
embeddings = torch.cat(vectors).numpy()
print("embeddings:", embeddings.shape)

## 6) Simplify with PCA

PCA squeezes each 1280-number embedding down to 128 -- faster and less noisy.

In [ ]:
reduced = PCA(n_components=128).fit_transform(embeddings)

## 7) Search for the 5 most similar images

In [ ]:
query = 0                                       # try other numbers (0-799)!
scores = cosine_similarity([reduced[query]], reduced)[0]
top5 = scores.argsort()[::-1][1:6]              # most similar (skip the query itself)

## 8) Show the query and its look-alikes

In [ ]:
plt.figure(figsize=(12, 3))
plt.subplot(1, 6, 1); plt.imshow(shown[idx[query]][0]); plt.title("query"); plt.axis("off")
for j, k in enumerate(top5):
    plt.subplot(1, 6, j + 2); plt.imshow(shown[idx[k]][0]); plt.title(f"#{j+1}"); plt.axis("off")
plt.show()

## Try it yourself

- Change `query` to any number from 0-799 and re-run steps 7-8.
- The model was never trained to *search* -- it just understands images well
  enough that similar ones land near each other. That's the power of transfer
  learning.

### *Contributed By: Sattam Altwaim*